# CNN Experiments

- Run the CNN architectures, ResNet-50 andtrained with and without background randomisation augmentation, across multiple seeds.

# Importing libraries

In [1]:
import pathlib
import json
import torch
import numpy as np
import random
import pandas as pd
import time

from cory_data_wrapper import *
from cory_utils import *

# Select device

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

Using cuda


# Create constants and create paths

In [3]:
# Make sure that the directory exists
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
SEEDS = [0, 339, 1, 2 ,3]
CNN_ARCHITECTURES = ["resnet50"]

DATA_DIR = pathlib.Path("data")
DATA_ROUTE = DATA_DIR / "rebuild" / "in9_train"
VAL_ROUTE = DATA_DIR / "rebuild" /"in9_val"
BOUNDING_BOX_PATH = DATA_DIR / "LOC_train_solution.csv"
TEST_INSTANCES = DATA_DIR / "bg_challenge"
RESULTS = pathlib.Path("./results")
TENSORBOARD_DIR = pathlib.Path("./runs")
FIGURES = pathlib.Path("./figures")
RESULTS.mkdir(exist_ok=True)
TENSORBOARD_DIR.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

# Load the data

In [4]:
# Load data
bboxs = load_bbox_csv(BOUNDING_BOX_PATH)
base_train_dataset = IN9BoundingBoxDataset(DATA_ROUTE, bboxs, image_size=IMAGE_SIZE)
train_class_to_idx = base_train_dataset.class_to_idx


# print the instances and num of classes
print(f"Loaded {len(base_train_dataset)} images", "with" , len(base_train_dataset.classes), "classes")

Loaded 69120 images with 9 classes


# Split the data (Train, Validation)

In [5]:
train_subset = base_train_dataset
# Create data loaders for all sets, for faster loading and shuffling and batches
# Create the validation loader
val_dataset_normalised = BackgroundRandomizationDataset(VAL_ROUTE, augmentation_strength=0.0, seed=0)
val_loader = make_loader(val_dataset_normalised, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Loaded {len(val_dataset_normalised)} validation images")

# make loaders for the test set, one for each variant
test_variant_loaders = {
    variant: make_loader(
        fix_test_labels(
            ImageFolderEval(TEST_INSTANCES / variant / "val", image_size=IMAGE_SIZE),train_class_to_idx
        ),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )
    for variant in ["original", "no_fg", "mixed_same", "mixed_rand"]
}
print("Test variants loaders created:", list(test_variant_loaders.keys()))

Test variants loaders created: ['original', 'no_fg', 'mixed_same', 'mixed_rand']


# Perform Learning Rate Search

- Perform a small learning rate search on Validation Accuracy
- Train for 3 epochs on 3 different learning rates.

In [6]:
# Create the dictionary for best Learning-Rate
BEST_LR = {
    "resnet50": None
}
# added the value so it didnt run a second time
best_lr_per_arch = {
    }
# loop through all architectures and perform the Hyper Parameter search for each architecture
for arch_name in CNN_ARCHITECTURES:
    # Check to ensure that it only runs if LR not known,
    if BEST_LR[arch_name] is not None:
        best_lr_per_arch[arch_name] = BEST_LR[arch_name]
        continue

    print(f"\n{'-'*60}\n Hyper Parameter Search: {arch_name} \n{'-'*60}")

    search_dataset = BackgroundRandomizationDataset(train_subset, augmentation_strength=0.0, seed=0)
    search_train_loader = make_loader(search_dataset,batch_size=BATCH_SIZE, shuffle=True, num_workers=12, persistent_workers=True)

    best_lr, hyperparameter_trials = hyperparameter_search(
        lambda a = arch_name: CNNModel(a, num_classes=9, pretrained=True),
        search_train_loader, val_loader, device,
        learning_rates=(2.5e-5, 5e-5, 1e-4), search_epochs=3,
        log_dir=TENSORBOARD_DIR / "hpsearch" / arch_name
    )
    best_lr_per_arch[arch_name] = best_lr
    print(f"{arch_name} best LR: {best_lr:}")

print("\nBest LR per architecture:", best_lr_per_arch)


Best LR per architecture: {'resnet50': 3e-05}


# Architecture Train, Eval and Test Function

In [7]:
def run_cnn_experiment(arch_name, augmented, seed):
    """
    Run the CNN experiment for a specified architecture and seed.
    Stores the data in results folder

    @param arch_name: Name of the CNN architecture.
    @param augmented: Whether to use augmented or original data.
    @param seed: Seed for reproducibility.
    @return: A dictionary containing the results of the specified experiment.
    """
    data_type = "augmented" if augmented else "original"
    run_name = f"{arch_name}_{data_type}_{seed}"
    print(f"\n{'='*60}\n{run_name}\n{'='*60}")

    # If data is augmented, strength = 1 (swap) else 0 (do not swap background), swap the background with a random image's background
    train_dataset_for_run = BackgroundRandomizationDataset(
        train_subset, augmentation_strength=1.0 if augmented else 0.0, seed=seed
    )
    train_loader = make_loader(train_dataset_for_run, batch_size=BATCH_SIZE, shuffle=True, num_workers=12, persistent_workers=True)

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    # Define the model and move it to the device
    model = CNNModel(arch_name, num_classes=9, pretrained=True).to(device)

    start_time = time.time()
    # Train the model and store the results
    trained_model, history = train_model(
        model, train_loader, val_loader, device, epochs=EPOCHS, early_stopping_patience=3, model_name=run_name, lr=best_lr_per_arch[arch_name], log_dir=TENSORBOARD_DIR / run_name
    )
    training_time_in_secs = time.time() - start_time

    # Save the model weights
    weights_path = RESULTS / f"{run_name}_weights.pt"
    torch.save(trained_model.state_dict(), weights_path)

    variant_accuracies = evaluate_on_variants(trained_model, test_variant_loaders, device)
    # compute the reliance metrics
    reliance = compute_reliance_metrics(variant_accuracies, num_classes=9)
    print(f"Test variants: {variant_accuracies}")
    print(f"BG-Gap: {reliance['bg_gap']:.4f}, No-FG: {reliance['no_fg_accuracy']:.4f}, Original: {reliance['original_accuracy']:.4f}")
    # Store the results in a dictionary
    results = {
        "architecture" : arch_name,
        "data_type" : data_type,
        "seed" : seed,
        "variant_accuracies" : variant_accuracies,
        "reliance_metrics" : reliance,
        "final_val_accuracy" : history["val_accuracy"][-1],
        "epochs_run" : len(history["val_accuracy"]),
        "training_time_in_secs" : training_time_in_secs,
        "best_val_accuracy" : max(history["val_accuracy"]),
    }
    # Save the dadta
    results_path = RESULTS / "cnn_results.json"
    results_each_run = {}
    if results_path.exists():
        with open(results_path, "r") as f:
            for r in json.load(f):
                key = f"{r['architecture']}_{r['data_type']}_{r['seed']}"
                results_each_run[key] = r
    results_each_run[run_name] = results
    with open(results_path, "w") as f:
        json.dump(list(results_each_run.values()), f, indent=2)

    return results

# Run Each model and Seed

### ResNet50 (No Augmentation)

In [8]:
run_cnn_experiment("resnet50", augmented=False, seed=0)


resnet50_original_0
[resnet50_original_0] Epoch 1/10: train_loss=0.7798  train_acc=0.7800 val_loss=0.0493  val_acc=0.9868
[resnet50_original_0] Epoch 2/10: train_loss=0.0590  train_acc=0.9828 val_loss=0.0379  val_acc=0.9896
[resnet50_original_0] Epoch 3/10: train_loss=0.0192  train_acc=0.9946 val_loss=0.0437  val_acc=0.9903
[resnet50_original_0] Epoch 4/10: train_loss=0.0099  train_acc=0.9972 val_loss=0.0480  val_acc=0.9887
[resnet50_original_0] Epoch 5/10: train_loss=0.0052  train_acc=0.9983 val_loss=0.0551  val_acc=0.9897
[resnet50_original_0] Epoch 6/10: train_loss=0.0033  train_acc=0.9990 val_loss=0.0469  val_acc=0.9910
[resnet50_original_0] Epoch 7/10: train_loss=0.0019  train_acc=0.9995 val_loss=0.0493  val_acc=0.9910
[resnet50_original_0] Epoch 8/10: train_loss=0.0016  train_acc=0.9995 val_loss=0.0483  val_acc=0.9907
[resnet50_original_0] Epoch 9/10: train_loss=0.0005  train_acc=0.9999 val_loss=0.0504  val_acc=0.9916
[resnet50_original_0] Epoch 10/10: train_loss=0.0004  train_a

{'architecture': 'resnet50',
 'data_type': 'original',
 'seed': 0,
 'variant_accuracies': {'original': 0.9869135802469136,
  'no_fg': 0.5995061728395061,
  'mixed_same': 0.9530864197530864,
  'mixed_rand': 0.9125925925925926},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.5995061728395061,
  'bg_gap': 0.04049382716049377,
  'original_accuracy': 0.9869135802469136},
 'final_val_accuracy': 0.9917534722222222,
 'epochs_run': 10,
 'training_time_in_secs': 1486.5889506340027}

In [9]:
run_cnn_experiment("resnet50", augmented=False, seed=339)


resnet50_original_339
[resnet50_original_339] Epoch 1/10: train_loss=0.7868  train_acc=0.7748 val_loss=0.0493  val_acc=0.9878
[resnet50_original_339] Epoch 2/10: train_loss=0.0598  train_acc=0.9832 val_loss=0.0363  val_acc=0.9891
[resnet50_original_339] Epoch 3/10: train_loss=0.0198  train_acc=0.9944 val_loss=0.0416  val_acc=0.9899
[resnet50_original_339] Epoch 4/10: train_loss=0.0093  train_acc=0.9969 val_loss=0.0504  val_acc=0.9897
[resnet50_original_339] Epoch 5/10: train_loss=0.0053  train_acc=0.9985 val_loss=0.0521  val_acc=0.9877
[resnet50_original_339] Epoch 6/10: train_loss=0.0035  train_acc=0.9991 val_loss=0.0525  val_acc=0.9905
[resnet50_original_339] Epoch 7/10: train_loss=0.0018  train_acc=0.9994 val_loss=0.0571  val_acc=0.9900
[resnet50_original_339] Epoch 8/10: train_loss=0.0015  train_acc=0.9996 val_loss=0.0538  val_acc=0.9899
[resnet50_original_339] Epoch 9/10: train_loss=0.0008  train_acc=0.9998 val_loss=0.0549  val_acc=0.9906
[resnet50_original_339] Epoch 10/10: trai

{'architecture': 'resnet50',
 'data_type': 'original',
 'seed': 339,
 'variant_accuracies': {'original': 0.988641975308642,
  'no_fg': 0.659753086419753,
  'mixed_same': 0.9560493827160493,
  'mixed_rand': 0.914074074074074},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.659753086419753,
  'bg_gap': 0.041975308641975295,
  'original_accuracy': 0.988641975308642},
 'final_val_accuracy': 0.9910300925925926,
 'epochs_run': 10,
 'training_time_in_secs': 1456.9788196086884}

In [10]:
run_cnn_experiment("resnet50", augmented =False, seed = 1)


resnet50_original_1
[resnet50_original_1] Epoch 1/10: train_loss=0.7663  train_acc=0.7926 val_loss=0.0534  val_acc=0.9854
[resnet50_original_1] Epoch 2/10: train_loss=0.0572  train_acc=0.9836 val_loss=0.0370  val_acc=0.9884
[resnet50_original_1] Epoch 3/10: train_loss=0.0185  train_acc=0.9946 val_loss=0.0372  val_acc=0.9893
[resnet50_original_1] Epoch 4/10: train_loss=0.0113  train_acc=0.9971 val_loss=0.0489  val_acc=0.9890
[resnet50_original_1] Epoch 5/10: train_loss=0.0052  train_acc=0.9984 val_loss=0.0473  val_acc=0.9905
[resnet50_original_1] Epoch 6/10: train_loss=0.0032  train_acc=0.9990 val_loss=0.0454  val_acc=0.9910
[resnet50_original_1] Epoch 7/10: train_loss=0.0017  train_acc=0.9995 val_loss=0.0457  val_acc=0.9915
[resnet50_original_1] Epoch 8/10: train_loss=0.0013  train_acc=0.9996 val_loss=0.0483  val_acc=0.9918
[resnet50_original_1] Epoch 9/10: train_loss=0.0005  train_acc=0.9999 val_loss=0.0503  val_acc=0.9922
[resnet50_original_1] Epoch 10/10: train_loss=0.0007  train_a

{'architecture': 'resnet50',
 'data_type': 'original',
 'seed': 1,
 'variant_accuracies': {'original': 0.9876543209876543,
  'no_fg': 0.6723456790123457,
  'mixed_same': 0.9619753086419753,
  'mixed_rand': 0.9128395061728395},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6723456790123457,
  'bg_gap': 0.04913580246913585,
  'original_accuracy': 0.9876543209876543},
 'final_val_accuracy': 0.9914641203703703,
 'epochs_run': 10,
 'training_time_in_secs': 1437.2147190570831}

In [ ]:
run_cnn_experiment("resnet50", augmented =False, seed = 2)


In [ ]:
run_cnn_experiment("resnet50", augmented =False, seed = 3)


### ResNet50 (Augmentation)


In [8]:
run_cnn_experiment("resnet50", augmented=True, seed=0)


resnet50_augmented_0
[resnet50_augmented_0] Epoch 1/10: train_loss=1.0105  train_acc=0.6829 val_loss=0.1394  val_acc=0.9758
[resnet50_augmented_0] Epoch 2/10: train_loss=0.1671  train_acc=0.9497 val_loss=0.1220  val_acc=0.9769
[resnet50_augmented_0] Epoch 3/10: train_loss=0.0958  train_acc=0.9722 val_loss=0.1036  val_acc=0.9737
[resnet50_augmented_0] Epoch 4/10: train_loss=0.0624  train_acc=0.9825 val_loss=0.1301  val_acc=0.9690
[resnet50_augmented_0] Epoch 5/10: train_loss=0.0445  train_acc=0.9877 val_loss=0.1222  val_acc=0.9693
[resnet50_augmented_0] Early stopping at epoch 5
Test variants: {'original': 0.98, 'no_fg': 0.6091358024691358, 'mixed_same': 0.9577777777777777, 'mixed_rand': 0.9437037037037037}
BG-Gap: 0.0141, No-FG: 0.6091, Original: 0.9800


{'architecture': 'resnet50',
 'data_type': 'augmented',
 'seed': 0,
 'variant_accuracies': {'original': 0.98,
  'no_fg': 0.6091358024691358,
  'mixed_same': 0.9577777777777777,
  'mixed_rand': 0.9437037037037037},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6091358024691358,
  'bg_gap': 0.014074074074074017,
  'original_accuracy': 0.98},
 'final_val_accuracy': 0.9693287037037037,
 'epochs_run': 5,
 'training_time_in_secs': 2686.005913257599}

In [9]:
run_cnn_experiment("resnet50", augmented=True, seed=339)


resnet50_augmented_339
[resnet50_augmented_339] Epoch 1/10: train_loss=1.0185  train_acc=0.6812 val_loss=0.1499  val_acc=0.9802
[resnet50_augmented_339] Epoch 2/10: train_loss=0.1655  train_acc=0.9508 val_loss=0.1196  val_acc=0.9755
[resnet50_augmented_339] Epoch 3/10: train_loss=0.0948  train_acc=0.9726 val_loss=0.1156  val_acc=0.9754
[resnet50_augmented_339] Epoch 4/10: train_loss=0.0636  train_acc=0.9821 val_loss=0.1358  val_acc=0.9732
[resnet50_augmented_339] Early stopping at epoch 4
Test variants: {'original': 0.9790123456790123, 'no_fg': 0.594320987654321, 'mixed_same': 0.9530864197530864, 'mixed_rand': 0.9325925925925926}
BG-Gap: 0.0205, No-FG: 0.5943, Original: 0.9790


{'architecture': 'resnet50',
 'data_type': 'augmented',
 'seed': 339,
 'variant_accuracies': {'original': 0.9790123456790123,
  'no_fg': 0.594320987654321,
  'mixed_same': 0.9530864197530864,
  'mixed_rand': 0.9325925925925926},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.594320987654321,
  'bg_gap': 0.02049382716049375,
  'original_accuracy': 0.9790123456790123},
 'final_val_accuracy': 0.9732349537037037,
 'epochs_run': 4,
 'training_time_in_secs': 2171.719069957733}

In [10]:
run_cnn_experiment("resnet50", augmented=True, seed = 1)


resnet50_augmented_1
[resnet50_augmented_1] Epoch 1/10: train_loss=0.9873  train_acc=0.6983 val_loss=0.1305  val_acc=0.9786
[resnet50_augmented_1] Epoch 2/10: train_loss=0.1663  train_acc=0.9508 val_loss=0.1084  val_acc=0.9784
[resnet50_augmented_1] Epoch 3/10: train_loss=0.0965  train_acc=0.9715 val_loss=0.1144  val_acc=0.9737
[resnet50_augmented_1] Epoch 4/10: train_loss=0.0617  train_acc=0.9825 val_loss=0.1327  val_acc=0.9695
[resnet50_augmented_1] Early stopping at epoch 4
Test variants: {'original': 0.974320987654321, 'no_fg': 0.6212345679012345, 'mixed_same': 0.9488888888888889, 'mixed_rand': 0.9274074074074075}
BG-Gap: 0.0215, No-FG: 0.6212, Original: 0.9743


{'architecture': 'resnet50',
 'data_type': 'augmented',
 'seed': 1,
 'variant_accuracies': {'original': 0.974320987654321,
  'no_fg': 0.6212345679012345,
  'mixed_same': 0.9488888888888889,
  'mixed_rand': 0.9274074074074075},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6212345679012345,
  'bg_gap': 0.021481481481481435,
  'original_accuracy': 0.974320987654321},
 'final_val_accuracy': 0.9694733796296297,
 'epochs_run': 4,
 'training_time_in_secs': 2168.7140135765076}

In [ ]:
run_cnn_experiment("resnet50", augmented=True, seed = 2)


In [ ]:
run_cnn_experiment("resnet50", augmented=True, seed = 3)


# Short Results Summary

In [11]:
results_path = RESULTS / "cnn_results.json"
with open(results_path, "r") as f:
    results = json.load(f)

rows = []
for r in results:
    row = {"architecture": r["architecture"], "data_type": r["data_type"], "seed": r["seed"], "best_val_accuracy": r["best_val_accuracy"], "epochs_run": r["epochs_run"]}
    row.update(r["reliance_metrics"])
    row["original_accuracy"] = r["variant_accuracies"].get("original")
    rows.append(row)

df = pd.DataFrame(rows)
summary = df.groupby(["architecture", "data_type"]).agg(
    mean_bg_gap =("bg_gap", "mean"), std_bg_gap =("bg_gap", "std"),
    mean_no_fg =("no_fg_accuracy", "mean"), std_no_fg =("no_fg_accuracy", "std"),
    mean_original =("original_accuracy", "mean"), std_original =("original_accuracy", "std")
)
summary

mean_bg_gap  std_bg_gap  mean_no_fg  std_no_fg  \
architecture data_type                                                   
resnet50     augmented     0.018683    0.004022    0.608230   0.013480   
             original      0.043868    0.004622    0.643868   0.038931   

                        mean_original  std_original  
architecture data_type                               
resnet50     augmented       0.977778      0.003034  
             original        0.987737      0.000867

# Show TensorBoard

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir ./runs

# GradCam

In [44]:
from cory_gradcam import get_resnet50_gradcam
from PIL import Image
import pathlib
import torch

model = CNNModel("resnet50", num_classes=9, pretrained=False)
# Print out the parameter count
print(f"ResNet50 Parameter Count: {sum(p.numel() for p in model.parameters()):,}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS = pathlib.Path("./results")

mixed_rand_test = fix_test_labels(ImageFolderEval(TEST_INSTANCES / "mixed_rand" / "val", image_size=IMAGE_SIZE), train_class_to_idx)
def perform_gradcam(augmented=False, img_index = 339):
    if augmented:
        model.load_state_dict(torch.load(RESULTS / f"resnet50_augmented_0_weights.pt"))
    else:
        model.load_state_dict(torch.load(RESULTS / f"resnet50_original_0_weights.pt"))
    model.to(device)
    image, label = mixed_rand_test[img_index]

    overlay, gradcam, predicted = get_resnet50_gradcam(model, image, device)
    print(f"True label: {label}, Predicted label: {predicted}")
    Image.fromarray(overlay).save(FIGURES/f"resnet50_gradcam_{'augmented' if augmented else 'original'}_{img_index}.png")
    Image.fromarray((gradcam * 255).astype(np.uint8)).save(FIGURES/f"resnet50_gradcam_{'augmented' if augmented else 'original'}heatmap_{img_index}.png")

perform_gradcam(augmented=False, img_index=2282)
perform_gradcam(augmented=True, img_index=2282)
Image.open(mixed_rand_test.samples[2282][0]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE)).save(FIGURES/"test_image.png")

ResNet50 Parameter Count: 23,526,473
True label: 4, Predicted label: 0
True label: 4, Predicted label: 4


# Inference time counter

In [37]:
import torch.utils.benchmark as benchmark
# https://docs.pytorch.org/docs/2.14/benchmark_utils.html
# https://docs.pytorch.org/tutorials/recipes/recipes/benchmark.html

def benchmark_model(model,name):
    model=model.cuda().eval()
    x = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device = "cuda")
    torch.cuda.reset_peak_memory_stats()
    timer = benchmark.Timer(stmt="with torch.no_grad(): model(x)", globals={"model": model, "x": x, "torch": torch})
    result = timer.blocked_autorange()
    print(f"{name}: took {result.median*1000} ms, with a peak memory usage of: {torch.cuda.max_memory_allocated() / 1e6} MegaBytes")

benchmark_model(CNNModel("resnet50", num_classes=9, pretrained=True), "resnet50")

ResNet50 took 0.025565716000273825 seconds, with a peak memory of: 665,154,048 bytes
